In [1]:
from google.colab import drive
# This will prompt you for authorization to mount your Google Drive
try:
    drive.mount('/content/drive', force_remount=True)
    print("\n[SUCCESS] Google Drive successfully mounted today!")
except Exception as e:
    print(f"\n[ERROR] Mount failed: {e}")

Mounted at /content/drive

[SUCCESS] Google Drive successfully mounted today!


In [ ]:
#Stock Specific File Creation
from google.colab import drive
import os
import subprocess
import time
import urllib.request
import json
import glob
from datetime import datetime, timedelta

# 1. Mount Google Drive safely
if not os.path.exists('/content/drive/MyDrive'):
    try:
        drive.mount('/content/drive')
    except Exception as e:
        print(f"Mounting failed or already mounted: {e}")
else:
    print("Google Drive is already mounted.")

# 2. Define target directory in Google Drive
download_path = "/content/drive/MyDrive/equitydata2/sec_bhavdata_full_1700Days/"
os.makedirs(download_path, exist_ok=True)

# 3. Analyze existing files to find oldest and newest dates
print("Scanning existing files in target directory...")
existing_files = glob.glob(os.path.join(download_path, "sec_bhavdata_full_*.csv"))

if existing_files:
    dates = []
    for filepath in existing_files:
        filename = os.path.basename(filepath)
        # Extract the date part: 'sec_bhavdata_full_DDMMYYYY.csv' -> 'DDMMYYYY'
        date_part = filename.replace("sec_bhavdata_full_", "").replace(".csv", "")
        try:
            file_date = datetime.strptime(date_part, "%d%m%Y")
            dates.append(file_date)
        except ValueError:
            continue

    if dates:
        oldest_date = min(dates)
        newest_date = max(dates)
        print(f"-> Found {len(dates)} valid CSV files in your Google Drive.")
        print(f"-> Oldest file date: {oldest_date.strftime('%d-%m-%Y')}")
        print(f"-> Newest file date: {newest_date.strftime('%d-%m-%Y')}")
    else:
        print("-> Found files, but none match valid/realistic date formats.")
else:
    print("-> No existing files found in directory.")

# 4. Create the list of URLs to download
file_list_path = "/tmp/sec_bhavdata_full_1700Days.txt"
base_url = "https://nsearchives.nseindia.com/products/content/sec_bhavdata_full_"

# Try to get the actual date dynamically, else default to system clock
print("\nFetching current date...")
real_today = None
time_apis = [
    "http://worldtimeapi.org/api/ip",
    "https://worldtimeapi.org/api/timezone/Etc/UTC"
]

for api_url in time_apis:
    try:
        req = urllib.request.Request(api_url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=5) as response:
            data = json.loads(response.read().decode())
            api_datetime_str = data["datetime"][:10]
            real_today = datetime.strptime(api_datetime_str, "%Y-%m-%d")
            print(f"Real-world date successfully retrieved via API: {real_today.strftime('%d-%m-%Y')}")
            break
    except Exception as e:
        continue

if not real_today:
    # Use the system clock directly without enforcing pre-2025 overrides
    real_today = datetime.now()
    print(f"API failed. Using local system clock: {real_today.strftime('%d-%m-%Y')}")

print("Generating URL list starting backward from current date...")
urls = []

for n in range(1700):
    target_date = real_today - timedelta(days=n)

    # Exclude Saturdays and Sundays since NSE is closed
    if target_date.weekday() in [5, 6]:
        continue

    date_str = target_date.strftime("%d%m%Y")
    urls.append(f"{base_url}{date_str}.csv")

with open(file_list_path, "w") as f:
    f.write("\n".join(urls) + "\n")

# 5. Download files directly to Google Drive
print("Starting download process...")
with open(file_list_path, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        filename = line.split("/")[-1]
        target_file = os.path.join(download_path, filename)

        # SKIP files that already exist, but DO NOT stop processing the rest of the list
        if os.path.exists(target_file):
            # print(f"File '{filename}' already exists. Skipping.")
            continue

        print(f"Downloading: {filename}")

        # Use curl to download the file with appropriate headers
        curl_command = [
            "curl", "-s", line,
            "-H", "Upgrade-Insecure-Requests: 1",
            "-H", "User-Agent: Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "-H", 'sec-ch-ua: "Not_A Brand";v="8", "Chromium";v="120", "Google Chrome";v="120"',
            "-H", "sec-ch-ua-mobile: ?0",
            "-H", 'sec-ch-ua-platform: "macOS"',
            "--compressed",
            "--output", target_file
        ]

        subprocess.run(curl_command)
        # Sleep to avoid rate limits
        time.sleep(1)

print("Download process finished.")

Google Drive is already mounted.
Scanning existing files in target directory...
-> No existing files found in directory.

Fetching current date...
API failed. Using local system clock: 11-11-2026
Generating URL list starting backward from current date...
Starting download process...
Downloading: sec_bhavdata_full_11112026.csv
Downloading: sec_bhavdata_full_10112026.csv
Downloading: sec_bhavdata_full_09112026.csv
Downloading: sec_bhavdata_full_06112026.csv
Downloading: sec_bhavdata_full_05112026.csv
Downloading: sec_bhavdata_full_04112026.csv
Downloading: sec_bhavdata_full_03112026.csv
Downloading: sec_bhavdata_full_02112026.csv
Downloading: sec_bhavdata_full_30102026.csv
Downloading: sec_bhavdata_full_29102026.csv
Downloading: sec_bhavdata_full_28102026.csv
